In [1]:
!python -m pip install pandas corus nltk scikit-learn seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 3.1 MB/s eta 0:00:00


Были проблемы с библиотеками которые завязаны на curos, поэтому просто через curl скачиваем

In [2]:
!curl -L -O https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  502M  100  502M    0     0   202M      0  0:00:02  0:00:02 --:--:--  245M


In [3]:
!pip install nltk
import nltk

In [4]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
import re
import os
import urllib.request

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns
import numpy as np
from corus import load_lenta

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

In [6]:
import random

# Фиксируем воспроизводимость результатов

In [7]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [8]:
base_dir = 'lenta-ru-news.csv.gz'
records = load_lenta(base_dir)

data =[]

# берем топики нужные
for record in records:
    data.append({
        'title': record.title,
        'text': record.text,
        'topic': record.topic
    })

df = pd.DataFrame(data)

In [9]:
df.head()

,title,text,topic
0,Названы регионы России с самой высокой смертно...,Вице-премьер по социальным вопросам Татьяна Го...,Россия
1,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт
2,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия
3,В США раскрыли сумму расходов на расследование...,С начала расследования российского вмешательст...,Мир
4,Хакеры рассказали о планах Великобритании зами...,Хакерская группировка Anonymous опубликовала н...,Мир


In [10]:
!pip install pymorphy3
import pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 83.5 MB/s eta 0:00:00


In [11]:
from pymorphy3 import MorphAnalyzer

In [12]:
ru_stopwords = set(stopwords.words('russian'))

In [13]:
stemmer = SnowballStemmer(language='russian')

In [14]:
morph = MorphAnalyzer()

re_url = re.compile(r'http\S+|www\S+')
re_non_alpha = re.compile(r'[^а-яА-ЯёЁ\s]')

def preprocess_text(text):

    if not isinstance(text, str):
        return ""
    text = text.lower()

    text = re_url.sub('', text)

    text = re_non_alpha.sub(' ', text)

    tokens = []
    for word in text.split():
        if word and word not in ru_stopwords:
            # Стемминг слова, пробовал лемматизацию но там не дождался обработки
            stem = stemmer.stem(word)
            tokens.append(stem)

    return ' '.join(tokens)

In [15]:
# Объединим заголовок и текст для получения более полного контекста

df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

In [16]:
df.head()

,title,text,topic,content
0,Названы регионы России с самой высокой смертно...,Вице-премьер по социальным вопросам Татьяна Го...,Россия,Названы регионы России с самой высокой смертно...
1,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт,Австрия не представила доказательств вины росс...
2,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия,Обнаружено самое счастливое место на планете С...
3,В США раскрыли сумму расходов на расследование...,С начала расследования российского вмешательст...,Мир,В США раскрыли сумму расходов на расследование...
4,Хакеры рассказали о планах Великобритании зами...,Хакерская группировка Anonymous опубликовала н...,Мир,Хакеры рассказали о планах Великобритании зами...


In [17]:
df_clean, t = train_test_split(
    df,
    train_size=100000,
    random_state=RANDOM_STATE,
    stratify=df['topic']

)

df_clean = df_clean.reset_index(drop=True)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

окей чета какая то ошибка, исправляем ниже

In [18]:
counts = df['topic'].value_counts()
valid_topics = counts[counts >= 50].index

df_clean = df[df['topic'].isin(valid_topics)]

In [19]:
df_final, t = train_test_split(
    df_clean,
    train_size=100000,
    random_state=RANDOM_STATE,
    stratify=df_clean['topic']

)

df_final = df_final.reset_index(drop=True)

In [20]:
df_final.head()

,title,text,topic,content
0,Государственный департамент США не устраивают ...,"Государственный департамент США заявил, что за...",Мир,Государственный департамент США не устраивают ...
1,Южная Осетия будет просить Россию отменить эко...,Власти Южной Осетии намерены просить Москву от...,Бывший СССР,Южная Осетия будет просить Россию отменить эко...
2,На выходных в Москве прошла пушкинская Велоночь,С 27 на 28 сентября состоялась Восьмая Московс...,Дом,На выходных в Москве прошла пушкинская Велоноч...
3,В Лондоне при взрыве пострадал рабочий,"В центре Лондона 7 февраля произошел взрыв, со...",Мир,В Лондоне при взрыве пострадал рабочий В центр...
4,Польша начнет массовый снос советских памятников,Власти Польши планируют снести почти 30 памятн...,Мир,Польша начнет массовый снос советских памятник...


In [21]:
df_final['cleaned_text'] = df_final['content'].apply(preprocess_text)

In [22]:
X = df_final['cleaned_text']
y = df_final['topic']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

In [23]:
X

,cleaned_text
0,государствен департамент сша устраива бумажн п...
1,южн осет прос росс отмен экономическ санкц вла...
2,выходн москв прошл пушкинск велоноч сентябр со...
3,лондон взрыв пострада рабоч центр лондон февра...
4,польш начнет массов снос советск памятник влас...
...,...
99995,кгб белорусс обвин джозеф ке коммерческ шпиона...
99996,летн бабушк казахста покор интернет рэп житух ...
99997,медвед подготов закон увольнен связ утрат дове...
99998,собак науч наход детск порнограф запах правоох...


In [24]:
y

,topic
0,Мир
1,Бывший СССР
2,Дом
3,Мир
4,Мир
...,...
99995,Бывший СССР
99996,Интернет и СМИ
99997,Россия
99998,Интернет и СМИ


In [25]:
df_final.head()

,title,text,topic,content,cleaned_text
0,Государственный департамент США не устраивают ...,"Государственный департамент США заявил, что за...",Мир,Государственный департамент США не устраивают ...,государствен департамент сша устраива бумажн п...
1,Южная Осетия будет просить Россию отменить эко...,Власти Южной Осетии намерены просить Москву от...,Бывший СССР,Южная Осетия будет просить Россию отменить эко...,южн осет прос росс отмен экономическ санкц вла...
2,На выходных в Москве прошла пушкинская Велоночь,С 27 на 28 сентября состоялась Восьмая Московс...,Дом,На выходных в Москве прошла пушкинская Велоноч...,выходн москв прошл пушкинск велоноч сентябр со...
3,В Лондоне при взрыве пострадал рабочий,"В центре Лондона 7 февраля произошел взрыв, со...",Мир,В Лондоне при взрыве пострадал рабочий В центр...,лондон взрыв пострада рабоч центр лондон февра...
4,Польша начнет массовый снос советских памятников,Власти Польши планируют снести почти 30 памятн...,Мир,Польша начнет массовый снос советских памятник...,польш начнет массов снос советск памятник влас...


In [26]:
X_train

,cleaned_text
32519,бел медвед командор сед стал мистер ро руч бел...
44666,порошенк поведа удовольств грядущ встреч трамп...
91319,пермск депутат отказа увольня губернатор депут...
50578,грузинск авиакомпан выстав росс счет блокад ав...
38399,воронежск безбожник переименова луч поселк без...
...,...
6273,церков рождественск служб пришел кажд й москви...
41107,силовик рассказа пособник сред блогер главн уп...
25137,цаха хизбалл обменя удар гражданск объект изра...
18645,умер писател публицист владимир карп москв м г...


In [27]:
y_train

,topic
32519,Из жизни
44666,Бывший СССР
91319,Россия
50578,Бывший СССР
38399,Россия
...,...
6273,Россия
41107,Интернет и СМИ
25137,Мир
18645,Культура


Взяли предзагрузку и обработку из предыдущей дз, дальше топтать тропинки надо новые

# Word2vec


In [28]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 71.1 MB/s eta 0:00:00


In [29]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

In [30]:
train = [text.split() for text in X_train]

w2v = Word2Vec(
    sentences=train,
    vector_size=300,
    window=5,
    min_count=5,
    workers=4,
    seed=RANDOM_STATE
)

# все гиперпараметры взял дефолтные, можно другие попробовать, но я как всегда делаю это перед дедлайном

In [45]:
print(w2v.wv.index_to_key[:40])

['год', 'котор', 'сообща', 'росс', 'такж', 'эт', 'сво', 'компан', 'российск', 'заяв', 'процент', 'врем', 'стран', 'президент', 'дан', 'слов', 'нов', 'доллар', 'тысяч', 'человек', 'сша', 'перв', 'миллион', 'суд', 'дел', 'одн', 'ран', 'стал', 'москв', 'однак', 'рубл', 'сообщ', 'получ', 'глав', 'представител', 'друг', 'украин', 'мест', 'так', 'лет']


Проанализируем нормально ли наша моделька находит эмббединги на взгляд

In [38]:
print(w2v.wv.most_similar('год'))

[('го', 0.6695782542228699), ('середин', 0.6527577042579651), ('конц', 0.5905921459197998), ('начал', 0.5243188142776489), ('текущ', 0.5140240788459778), ('период', 0.4923253357410431), ('месяц', 0.4887068271636963), ('недел', 0.4468948245048523), ('включительн', 0.4154852628707886), ('конец', 0.4108661115169525)]


In [39]:
print(w2v.wv.most_similar('президент'))

[('президентск', 0.6300497651100159), ('госсекретар', 0.5701395273208618), ('посл', 0.520163357257843), ('спикер', 0.5199938416481018), ('посол', 0.4862595200538635), ('консул', 0.4704835116863251), ('губернатор', 0.46548956632614136), ('премьер', 0.4627959132194519), ('байд', 0.46081435680389404), ('мюнтеферинг', 0.4526115655899048)]


In [40]:
print(w2v.wv.most_similar('стран'))

[('государств', 0.5966249704360962), ('турц', 0.5561748147010803), ('лив', 0.5289770364761353), ('афганиста', 0.5271387100219727), ('белорусс', 0.5085036158561707), ('евросоюз', 0.496725469827652), ('ес', 0.4953077435493469), ('республик', 0.49483945965766907), ('кит', 0.4899286925792694), ('ирак', 0.4864296019077301)]


In [41]:
print(w2v.wv.doesnt_match(['год', 'президент', 'стран', 'сво']))

президент


Хм, странно что президент тут оказался лишним, но будем считать это ошибкой

In [49]:
print(w2v.wv.doesnt_match(['год', 'президент', 'стран', 'доллар']))

президент


Опять прездиент лишний, да что такое

In [50]:
print(w2v.wv.doesnt_match(['год', 'президент', 'стран', 'сообщ']))

сообщ


# Загрузим предобученые эмбединги navec и rusvectores

In [51]:
!curl -L -O https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 25.4M  100 25.4M    0     0  10.2M      0  0:00:02  0:00:02 --:--:-- 10.2M


In [54]:
!pip install navec

In [55]:
from navec import Navec

In [56]:
path_navec = 'navec_news_v1_1B_250K_300d_100q.tar'
navec = Navec.load(path_navec)

In [57]:
!curl -L -O http://vectors.nlpl.eu/repository/11/180.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  462M  100  462M    0     0  20.3M      0  0:00:22  0:00:22 --:--:-- 21.9M


In [58]:
!unzip 180.zip -d rusvectores_model

Archive:  180.zip
  inflating: rusvectores_model/README  
  inflating: rusvectores_model/meta.json  
  inflating: rusvectores_model/model.bin  
  inflating: rusvectores_model/model.txt  


In [59]:
path_rus = 'rusvectores_model/model.bin'

rusvec = KeyedVectors.load_word2vec_format(path_rus, binary=True)

In [62]:
print(list(rusvec.key_to_index.keys())[:10])

['так_ADV', 'быть_VERB', 'мочь_VERB', 'год_NOUN', 'человек_NOUN', 'xxxxxx_NUM', 'сказать_VERB', 'еще_ADV', 'один_NUM', 'говорить_VERB']


In [64]:
print(w2v.wv.index_to_key[:10])

['год', 'котор', 'сообща', 'росс', 'такж', 'эт', 'сво', 'компан', 'российск', 'заяв']


In [69]:
def get_text_embeddingwv(text, model, vector_size=300):
    vectors = [model.wv[token] for token in str(text).split() if token in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)

In [70]:
X_train_w2v = np.array([get_text_embeddingwv(text, w2v, 'w2v') for text in X_train])
X_val_w2v   = np.array([get_text_embeddingwv(text, w2v, 'w2v') for text in X_val])

In [71]:
def get_text_embedding(text, model, vector_size=300):
    vectors = [model[token] for token in str(text).split() if token in model]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)

In [72]:
X_train_navec = np.array([get_text_embedding(text, navec, 'navec') for text in X_train])
X_val_navec   = np.array([get_text_embedding(text, navec, 'navec') for text in X_val])

TypeError: 'str' object cannot be interpreted as an integer

In [73]:
# решил сделать универсальный эмбеддинг

Разбиваем текст на токены

Для каждого слова проверяем наличие в словаре

Если слово есть - берём его вектор

Вычисляем среднее арифметическое всех найденных векторов

In [74]:
# для каждой модели реализуем свой эмбеддинг а потом усредним по векторам
def get_text_embedding(text, model, emb_type='w2v', vector_size=300):
    tokens = str(text).split()
    vectors = []

    for token in tokens:
        if emb_type == 'w2v':
            if token in model.wv:
                vectors.append(model.wv[token])

        elif emb_type == 'navec':
            if token in model:
                vectors.append(model[token])

        elif emb_type == 'rusvectores':
            for pos in ['_NOUN', '_VERB', '_ADJ', '_PROPN', '_ADV']: # будем еще тег добавлять
                key = token + pos
                if key in model:
                    vectors.append(model[key])
                    break

    return np.mean(vectors, axis=0) if vectors else np.zeros(vector_size)

In [75]:
X_train_w2v = np.array([get_text_embedding(text, w2v, 'w2v') for text in X_train])
X_val_w2v   = np.array([get_text_embedding(text, w2v, 'w2v') for text in X_val])

In [76]:
X_train_navec = np.array([get_text_embedding(text, navec, 'navec') for text in X_train])
X_val_navec   = np.array([get_text_embedding(text, navec, 'navec') for text in X_val])

In [78]:
X_train_rusv = np.array([get_text_embedding(text, rusvec, 'rusvectores') for text in X_train])
X_val_rusv   = np.array([get_text_embedding(text, rusvec, 'rusvectores') for text in X_val])

In [79]:
w2v_LR = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1)
w2v_LR.fit(X_train_w2v, y_train)

pred_w2v = w2v_LR.predict(X_val_w2v)
print("Word2Vec")
print(classification_report(y_val, pred_w2v, zero_division=0))

Word2Vec
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         6
   69-я параллель       0.80      0.11      0.20        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.60      0.19      0.29       200
      Бывший СССР       0.79      0.74      0.76      1444
              Дом       0.80      0.78      0.79       588
         Из жизни       0.60      0.56      0.58       747
   Интернет и СМИ       0.72      0.65      0.68      1208
             Крым       0.40      0.11      0.17        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.84      0.86      0.85      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.77      0.81      0.79      3698
  Наука и техника       0.78      0.79      0.79      1437
      Путешествия       0.66      0.52      0.59       174
           Россия       0.73      0.80      0.

In [80]:
navec_LR = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1)

navec_LR.fit(X_train_navec, y_train)

pred_navec = navec_LR.predict(X_val_navec)
print("Navec")
print(classification_report(y_val, pred_navec, zero_division=0))

Navec
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         6
   69-я параллель       0.60      0.09      0.15        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.47      0.11      0.18       200
      Бывший СССР       0.74      0.67      0.70      1444
              Дом       0.79      0.73      0.76       588
         Из жизни       0.56      0.47      0.51       747
   Интернет и СМИ       0.68      0.60      0.64      1208
             Крым       0.00      0.00      0.00        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.82      0.84      0.83      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.73      0.77      0.75      3698
  Наука и техника       0.74      0.77      0.76      1437
      Путешествия       0.68      0.43      0.53       174
           Россия       0.69      0.78      0.73 

In [81]:
rusvec_LR = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1)
rusvec_LR.fit(X_train_rusv, y_train)

pred_rusvec = rusvec_LR.predict(X_val_rusv)
print("rusvectores")
print(classification_report(y_val, pred_rusvec, zero_division=0))

rusvectores
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         6
   69-я параллель       0.33      0.11      0.17        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.40      0.10      0.17       200
      Бывший СССР       0.70      0.60      0.64      1444
              Дом       0.75      0.66      0.70       588
         Из жизни       0.49      0.40      0.44       747
   Интернет и СМИ       0.63      0.55      0.59      1208
             Крым       0.27      0.17      0.21        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.79      0.81      0.80      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.71      0.74      0.73      3698
  Наука и техника       0.71      0.73      0.72      1437
      Путешествия       0.59      0.46      0.52       174
           Россия       0.66      0.77     

w2v выдал самые лучшие макро-f1, потом navec и заключительный rusvectores

думаю, что модель, обученная на релевантных данных (наши новости), всегда обгонит универсальную, если задача специфичная. А сложная структура векторов (как у rusvectores) требует более умного агрегирования, чем просто среднее

возьмем для улучшения модель w2v тогда, раз она самая лучшая для данной задачи

# Improve model w2v

In [82]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [84]:
tfidf = TfidfVectorizer(min_df=10, max_df=0.7)
tfidf.fit(X_train)

idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

In [85]:
def get_weighted_embedding(text, model, idf_dict, emb_type='w2v', vector_size=300):
    tokens = str(text).split()
    vectors = []
    weights =[]

    for token in tokens:
        if token not in idf_dict:
            continue

        weight = idf_dict[token]

        if emb_type == 'w2v':
            if token in model.wv:
                vectors.append(model.wv[token])
                weights.append(weight)

        elif emb_type == 'navec':
            if token in model.vocab:
                vectors.append(model[token])
                weights.append(weight)

        elif emb_type == 'rusvectores':
            for pos in['_NOUN', '_VERB', '_ADJ', '_PROPN', '_ADV']:
                if token + pos in model:
                    vectors.append(model[token + pos])
                    weights.append(weight)
                    break

    if len(vectors) > 0:
        return np.average(vectors, axis=0, weights=weights)
    else:
        return np.zeros(vector_size)

In [87]:
X_train_weighted = np.array([get_weighted_embedding(text, w2v, idf_dict, emb_type='w2v') for text in X_train])
X_val_weighted   = np.array([get_weighted_embedding(text, w2v, idf_dict, emb_type='w2v') for text in X_val])

In [88]:
weighted_LR = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1)
weighted_LR.fit(X_train_weighted, y_train)

print("tfidf")
print(classification_report(y_val, weighted_LR.predict(X_val_weighted), zero_division=0))

tfidf
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         6
   69-я параллель       0.50      0.09      0.15        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.54      0.16      0.25       200
      Бывший СССР       0.79      0.75      0.77      1444
              Дом       0.81      0.77      0.79       588
         Из жизни       0.59      0.56      0.58       747
   Интернет и СМИ       0.72      0.66      0.69      1208
             Крым       0.50      0.11      0.18        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.85      0.87      0.86      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.77      0.81      0.79      3698
  Наука и техника       0.79      0.81      0.80      1437
      Путешествия       0.67      0.51      0.58       174
           Россия       0.73      0.80      0.76 

# Comparison Models

In [89]:

X_test_w2v = np.array([get_text_embedding(text, w2v, 'w2v') for text in X_test])
print("Word2Vec")
print(classification_report(y_test, w2v_LR.predict(X_test_w2v), zero_division=0))

Word2Vec
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.62      0.24      0.34        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.48      0.15      0.23       200
      Бывший СССР       0.78      0.73      0.75      1445
              Дом       0.79      0.73      0.76       588
         Из жизни       0.61      0.54      0.58       747
   Интернет и СМИ       0.71      0.67      0.69      1209
             Крым       0.33      0.06      0.10        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.84      0.84      0.84      1455
          Легпром       0.00      0.00      0.00         3
              Мир       0.77      0.81      0.79      3697
  Наука и техника       0.78      0.80      0.79      1438
      Путешествия       0.63      0.50      0.56       173
           Россия       0.74      0.80      0.

In [90]:
X_test_navec = np.array([get_text_embedding(text, navec, 'navec') for text in X_test])
print("Navec")
print(classification_report(y_test, navec_LR.predict(X_test_navec), zero_division=0))

Navec
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.67      0.18      0.28        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.37      0.09      0.14       200
      Бывший СССР       0.74      0.64      0.69      1445
              Дом       0.75      0.70      0.73       588
         Из жизни       0.58      0.48      0.53       747
   Интернет и СМИ       0.69      0.62      0.65      1209
             Крым       0.00      0.00      0.00        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.82      0.85      0.84      1455
          Легпром       0.00      0.00      0.00         3
              Мир       0.73      0.77      0.75      3697
  Наука и техника       0.77      0.78      0.77      1438
      Путешествия       0.60      0.47      0.53       173
           Россия       0.69      0.77      0.73 

In [91]:
X_test_rusv = np.array([get_text_embedding(text, rusvec, 'rusvectores') for text in X_test])
print("Rusvectores")
print(classification_report(y_test, rusvec_LR.predict(X_test_rusv), zero_division=0))

Rusvectores
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.58      0.32      0.42        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.42      0.10      0.17       200
      Бывший СССР       0.67      0.56      0.61      1445
              Дом       0.73      0.65      0.69       588
         Из жизни       0.53      0.44      0.48       747
   Интернет и СМИ       0.65      0.59      0.62      1209
             Крым       0.14      0.06      0.08        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.80      0.82      0.81      1455
          Легпром       0.00      0.00      0.00         3
              Мир       0.71      0.75      0.73      3697
  Наука и техника       0.73      0.74      0.74      1438
      Путешествия       0.57      0.45      0.50       173
           Россия       0.66      0.75     

In [92]:
X_test_weighted = np.array([get_weighted_embedding(text, w2v, idf_dict, emb_type='w2v') for text in X_test])
print("TFIDF")
print(classification_report(y_test, weighted_LR.predict(X_test_weighted), zero_division=0))

TFIDF
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         5
   69-я параллель       0.83      0.29      0.43        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.46      0.12      0.19       200
      Бывший СССР       0.79      0.72      0.75      1445
              Дом       0.81      0.74      0.77       588
         Из жизни       0.61      0.54      0.58       747
   Интернет и СМИ       0.73      0.67      0.70      1209
             Крым       0.33      0.06      0.10        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.84      0.85      0.85      1455
          Легпром       0.00      0.00      0.00         3
              Мир       0.77      0.82      0.79      3697
  Наука и техника       0.78      0.81      0.79      1438
      Путешествия       0.64      0.53      0.58       173
           Россия       0.74      0.80      0.77 

Самый худшая моделька это rusv, потом navec, w2v, tfidf

Скорее всего, если сделать другой препроцессинг данных или подобрать другие гиперпараметры то navec и rusv будут по результатам такие же как w2v

в целом tfidf ни чем почти ни отличается от w2v

